In [1]:
import numpy as np
import statsmodels.api as sm    
import pandas as pd
import matplotlib.pyplot as plt

industry = pd.read_csv('/Users/vincentfernandes/Documents/School/Quantitative Finance/Industry.csv')

Industry Check

In [2]:
industry.rename(columns = {'mdate' : 'Date', 'MktRf' : 'Market Excess'}, inplace = True)

industry['Date'] = pd.to_datetime(industry['Date'], format = '%Y%m')

cols = industry.columns.difference(['Date'])

for c in cols:
    industry[c] = (industry[c].astype(str).str.replace('%', '').astype(float) / 100)

print(cols)
 

Index(['Autos', 'Beer', 'Books', 'BusEq', 'Carry', 'Chems', 'Clths', 'Cnstr',
       'Coal', 'ElcEq', 'FabPr', 'Fin', 'Food', 'Games', 'Hlth', 'Hshld',
       'Market Excess', 'Meals', 'Mines', 'Oil', 'Other', 'Paper', 'Rf',
       'Rtail', 'Servs', 'Smoke', 'Steel', 'Telcm', 'Trans', 'Txtls', 'Util',
       'Whlsl'],
      dtype='object')


Compute Past 12-Month Average

In [5]:
# industry return columns (exclude Date, Rf, Market Excess)
industries = [c for c in industry.columns if c not in ['Date', 'Rf', 'Market Excess']]

# past 12-month average return, excluding current month (shift by 1)

past12 = industry[industries].shift(1).rolling(12).mean()


rankings = past12.rank(axis = 1, method = "first")

winners = rankings >= 16

print(rankings)

      Food  Beer  Smoke  Games  Books  Hshld  Clths  Hlth  Chems  Txtls  ...  \
0      NaN   NaN    NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN  ...   
1      NaN   NaN    NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN  ...   
2      NaN   NaN    NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN  ...   
3      NaN   NaN    NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN  ...   
4      NaN   NaN    NaN    NaN    NaN    NaN    NaN   NaN    NaN    NaN  ...   
...    ...   ...    ...    ...    ...    ...    ...   ...    ...    ...  ...   
1009   4.0  11.0   18.0   29.0   15.0   10.0   26.0   2.0   19.0   13.0  ...   
1010  10.0  18.0   24.0   30.0    6.0   16.0   20.0   5.0   25.0   15.0  ...   
1011   6.0  12.0   24.0   29.0   11.0    7.0   20.0   5.0   22.0   19.0  ...   
1012   8.0   5.0   20.0   30.0   13.0    3.0   18.0   4.0   27.0   23.0  ...   
1013   5.0   8.0   20.0   30.0   11.0    3.0   28.0   2.0   23.0   24.0  ...   

      Telcm  Servs  BusEq  Paper  Trans

Creating the excess

In [17]:
# winner portfolio total return each month (equal-weight across winners)
winner_ret = industry[industries].where(winners).mean(axis=1)

# winner portfolio excess return
winner_excess = winner_ret - industry['Rf']

# stats 
mean_excess = winner_excess.mean()
std_excess  = winner_excess.std(ddof=1)
sharpe_m    = mean_excess / std_excess
sharpe_a    = sharpe_m * np.sqrt(12)

print("Mean monthly excess:", mean_excess)
print("Std monthly excess :", std_excess)
print("Monthly Sharpe     :", sharpe_m)
print("Annual Sharpe      :", sharpe_a)

Mean monthly excess: 0.009528562874251497
Std monthly excess : 0.057069066943250514
Monthly Sharpe     : 0.16696545755210437
Annual Sharpe      : 0.5783853111784589
